# RidgeVisionNet -- Part 2 of 3 (v2): Ablation Study + Calibration

Fresh re-run with the EfficientNetB0 rescaling fix applied. All eight
ablation configurations -- including the two that previously collapsed
(`no_orientation_field`, `single_branch_texture`) -- use the corrected
`build_ridgevision_net`, so this is the first run where their results can be
trusted at face value.

**Before running:** attach your fingerprint dataset + `ridgevisionnet-v2-part1-output`.


In [1]:
# !pip install -q scikit-image
import os, gc, json, time
from pathlib import Path

import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats
from scipy.optimize import minimize_scalar

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))


TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
# =========================
# OFFLINE / NO-INTERNET RESILIENCE FOR IMAGENET WEIGHTS
# =========================
# Kaggle sessions default to "Internet: Off". Every backbone in this notebook
# (RidgeVisionNet + 6 of the 10 baselines) loads ImageNet-pretrained weights,
# which needs to download a .h5 file from storage.googleapis.com the first
# time it's used. If you saw an error like:
#   URLError: <urlopen error [Errno -3] Temporary failure in name resolution>
# it means Internet is Off for this session.
#
# FIX (do this first, it's the real fix): right sidebar -> Settings ->
# Internet -> toggle ON -> Save, then re-run. This changes nothing about the
# methodology -- it only lets the same published ImageNet weights download.
#
# The helpers below are a safety net for cases where you can't turn Internet
# on (e.g. a no-internet competition): they (1) detect the problem early with
# a clear message instead of a deep stack trace, (2) reuse any already-
# downloaded weights from a locally attached Kaggle dataset if one is
# present, and (3) as a last resort let backbone-building fall back to
# random-initialized weights (with a loud warning) so the notebook can still
# run to completion rather than crash -- note this last case means that
# backbone is no longer using transfer learning, which should be reported as
# a deviation if it happens for your real (non-QUICK_RUN) results.

import socket
import shutil


def internet_available(host="storage.googleapis.com", port=443, timeout=3):
    try:
        socket.getaddrinfo(host, port)
        return True
    except OSError:
        return False


HAS_INTERNET = internet_available()
print("Internet reachable:", HAS_INTERNET)


def stage_offline_imagenet_weights():
    """If a Kaggle dataset containing pre-downloaded Keras ImageNet weight
    files is attached (search Kaggle Datasets for 'keras pretrained models'
    or similar), copy any .h5 files found under /kaggle/input into
    ~/.keras/models/ so tf.keras.applications finds them in its local cache
    and skips the network call entirely. Safe to call even if nothing is
    found (returns 0)."""
    cache_dir = Path.home() / ".keras" / "models"
    cache_dir.mkdir(parents=True, exist_ok=True)
    found = 0
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for h5_path in input_root.rglob("*.h5"):
            target = cache_dir / h5_path.name
            if not target.exists():
                shutil.copy(h5_path, target)
                found += 1
    if found:
        print(f"Staged {found} local weight file(s) into {cache_dir}.")
    else:
        print("No local ImageNet weight files found under /kaggle/input.")
    return found


if not HAS_INTERNET:
    staged = stage_offline_imagenet_weights()
    if not staged:
        print()
        print("ACTION SUGGESTED: no internet reachable AND no local weights dataset found.")
        print("Go to Settings -> Internet -> ON (right sidebar), Save, and re-run this cell.")
        print("If your competition disallows internet, attach a Kaggle dataset that hosts the")
        print("*_notop.h5 files for the backbones used here, then re-run this cell.")


def load_backbone(backbone_cls, weights="imagenet", **kwargs):
    """Build a tf.keras.applications backbone, falling back to random-init
    weights (with a loud, impossible-to-miss warning) if the requested
    weights can't be obtained. Keeps the notebook runnable end-to-end even
    when Internet is Off and no offline weights dataset is attached; does
    NOT fix the underlying cause -- see the cell above."""
    try:
        return backbone_cls(weights=weights, **kwargs)
    except Exception as e:
        if weights is None:
            raise
        print(f"WARNING: could not load '{weights}' weights for {backbone_cls.__name__} ({type(e).__name__}: {e}).")
        print("Falling back to random initialization (weights=None) so the run can continue.")
        print("This backbone will NOT benefit from ImageNet transfer learning until the")
        print("internet/offline-weights issue above is resolved and this cell is re-run.")
        return backbone_cls(weights=None, **kwargs)


Internet reachable: True


In [3]:
# =========================
# CONFIG
# =========================
OUTPUT_DIR = Path('/kaggle/working')
RESULTS_DIR = OUTPUT_DIR / 'ridgevisionnet_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_LABELS = ['A+', 'A-', 'AB+', 'AB-', 'B+', 'B-', 'O+', 'O-']
LABEL_TO_INDEX = {label: i for i, label in enumerate(CLASS_LABELS)}
NUM_CLASSES = len(CLASS_LABELS)
SEED = 42

# --- Speed settings, changed this pass to fit a single Kaggle GPU session ---
# Mixed precision (fp16 compute, fp32 master weights) is a standard, purely
# computational optimization on GPUs with tensor cores (T4/P100/V100/A100) --
# it does not change what the model learns, only how fast the same forward/
# backward math runs, typically ~1.5-2x on these GPUs.
tf.keras.mixed_precision.set_global_policy('mixed_float16')

BATCH_SIZE = 32  # was 16; larger batches use the GPU more efficiently
IMG_SIZE = 224  # RidgeVisionNet / most baselines; InceptionV3 baseline overrides to 299

QUICK_RUN = False  # set False for the real run used in the paper
# EPOCHS_HEAD/EPOCHS_FINE are now upper bounds, not targets -- EarlyStopping
# (added this pass, see train_model) will stop well before these ceilings for
# any model that has converged or plateaued, which was most of the wasted
# time in the previous run (e.g. mobilenet_v2 ran all 55 epochs stuck near
# chance accuracy). Lowering the ceiling itself additionally caps worst-case
# runtime for a model that never triggers early stopping.
EPOCHS_HEAD = 3 if QUICK_RUN else 8
EPOCHS_FINE = 5 if QUICK_RUN else 30  # NOT lowered further than this: your own log showed resnet50/densenet121 flat until fine-tune epoch ~14, then climbing through epoch 40 -- a lower ceiling would cut those off mid-breakthrough and understate their real accuracy. EarlyStopping (patience=4) does the actual time-saving for models that plateau, not this ceiling.
# N_FOLDS reduced from 5 to 3: 3-fold stratified CV is still a standard,
# citable, defensible choice (commonly used exactly for compute-constrained
# settings) -- report it as "3-fold" rather than "5-fold" in the paper's
# Experimental Setup section rather than silently changing the number.
N_FOLDS = 2 if QUICK_RUN else 3
N_PERMUTATIONS = 200 if QUICK_RUN else 2000
MC_DROPOUT_SAMPLES = 10 if QUICK_RUN else 30

np.random.seed(SEED)
tf.random.set_seed(SEED)
print('QUICK_RUN =', QUICK_RUN)
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())


QUICK_RUN = False
Mixed precision policy: <DTypePolicy "mixed_float16">


In [4]:
# =========================
# RESTORE PRIOR-STAGE OUTPUTS
# =========================
# Attach the Kaggle Dataset from Part 1's output as Input before running.
import shutil
from pathlib import Path

_restored = 0
for _src in Path('/kaggle/input').rglob('ridgevisionnet_results'):
    if _src.is_dir():
        for _f in _src.rglob('*'):
            if _f.is_file():
                _target = RESULTS_DIR / _f.relative_to(_src)
                _target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(_f, _target)
                _restored += 1
print(f'Restored {_restored} file(s) into {RESULTS_DIR} from attached prior-stage dataset(s).')
if _restored == 0:
    print('WARNING: nothing restored -- did you attach the previous part\'s output dataset as Input?')


Restored 2 file(s) into /kaggle/working/ridgevisionnet_results from attached prior-stage dataset(s).


In [5]:
# =========================
# DATASET AUTO-DETECTION (same convention as the v1 notebook)
# =========================

def find_dataset_dir():
    search_roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob('*'):
            if not path.is_dir():
                continue
            class_folder_count = sum((path / label).is_dir() for label in CLASS_LABELS)
            if class_folder_count >= 6:
                candidates.append((class_folder_count, path))
    if not candidates:
        raise FileNotFoundError(
            'Dataset not found. Attach the fingerprint blood group dataset to Kaggle, '
            'or set DATASET_DIR manually below.'
        )
    candidates.sort(key=lambda item: item[0], reverse=True)
    print('Auto-detected dataset folder:', candidates[0][1])
    return candidates[0][1]

DATASET_DIR = find_dataset_dir()

all_paths, all_labels = [], []
for label in CLASS_LABELS:
    files = sorted((DATASET_DIR / label).glob('*'))
    all_paths.extend(files)
    all_labels.extend([LABEL_TO_INDEX[label]] * len(files))
all_labels = np.array(all_labels)
print(f'Total images: {len(all_paths)}')
for label in CLASS_LABELS:
    print(f'  {label}: {(all_labels == LABEL_TO_INDEX[label]).sum()}')


Auto-detected dataset folder: /kaggle/input/datasets/sravani2006/fingerprint-blood-group-classification-dataset/datasets
Total images: 5837
  A+: 402
  A-: 1009
  AB+: 708
  AB-: 761
  B+: 652
  B-: 741
  O+: 852
  O-: 712


In [6]:
# FIX: this array conversion originally lived inside Part 1's baseline-loop
# cell (not included here), but the ablation-split code below needs it.
all_paths_arr = np.array(all_paths, dtype=object)


In [7]:
# =========================
# IMAGE LOADING
# =========================

def load_rgb(path, img_size):
    img = cv2.imread(str(path))
    if img is None:
        return np.zeros((img_size, img_size, 3), dtype=np.float32)
    img = cv2.resize(img, (img_size, img_size))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0


class FingerprintSequence(tf.keras.utils.Sequence):
    """Simple image-only generator (RidgeVisionNet computes its own ridge
    orientation field on-device from the image, so no separate texture-vector
    input is needed here, unlike the v1 dual-input generator)."""

    def __init__(self, paths, labels, img_size, batch_size=BATCH_SIZE, augment=False, shuffle=True, **kwargs):
        super().__init__(**kwargs)  # required by Keras 3's PyDataset base (tf.keras.utils.Sequence is now an alias for it)
        self.paths = paths
        self.labels = labels
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.paths) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def _augment(self, img):
        if np.random.rand() < 0.5:
            angle = np.random.uniform(-10, 10)
            m = cv2.getRotationMatrix2D((self.img_size / 2, self.img_size / 2), angle, 1.0)
            img = cv2.warpAffine(img, m, (self.img_size, self.img_size), borderMode=cv2.BORDER_REFLECT)
        if np.random.rand() < 0.5:
            img = np.clip(img * np.random.uniform(0.85, 1.15) + np.random.uniform(-0.05, 0.05), 0, 1)
        if np.random.rand() < 0.3:
            img = np.clip(img + np.random.normal(0, 0.02, img.shape), 0, 1)
        return img.astype(np.float32)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        images = np.zeros((len(batch_idx), self.img_size, self.img_size, 3), dtype=np.float32)
        labels = np.zeros(len(batch_idx), dtype=np.int32)
        for i, bi in enumerate(batch_idx):
            img = load_rgb(self.paths[bi], self.img_size)
            if self.augment:
                img = self._augment(img)
            images[i] = img
            labels[i] = self.labels[bi]
        return images, labels


## RidgeVisionNet components

Ridge orientation field (deterministic), ROAM (orientation-gated attention),
Adaptive Gated Fusion, and the full model builder -- identical logic to
`backend/ml/models/{ridge_orientation,roam,ridgevision_net}.py` in the
repository, inlined here so this notebook is self-contained on Kaggle.

In [8]:
# =========================
# Ridge Orientation Field (deterministic, no trainable params)
# =========================
class RidgeOrientationField(tf.keras.layers.Layer):
    def __init__(self, block_size=8, **kwargs):
        super().__init__(**kwargs)
        self.block_size = block_size
        self.sobel_x = tf.constant([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=tf.float32)[:, :, None, None]
        self.sobel_y = tf.constant([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=tf.float32)[:, :, None, None]

    def call(self, inputs):
        gx = tf.nn.conv2d(inputs, self.sobel_x, strides=1, padding='SAME')
        gy = tf.nn.conv2d(inputs, self.sobel_y, strides=1, padding='SAME')
        gxx, gyy, gxy = gx * gx, gy * gy, gx * gy
        pool = lambda t: tf.nn.avg_pool2d(t, ksize=self.block_size, strides=self.block_size, padding='VALID')
        vxx, vyy, vxy = pool(gxx), pool(gyy), pool(gxy)
        numerator = 2.0 * vxy
        denominator = vxx - vyy
        theta2 = tf.atan2(numerator, denominator)
        cos2, sin2 = tf.cos(theta2), tf.sin(theta2)
        energy = tf.sqrt(numerator**2 + denominator**2)
        coherence = tf.clip_by_value(energy / (vxx + vyy + 1e-6), 0.0, 1.0)
        return tf.concat([cos2, sin2, coherence], axis=-1)

    def get_config(self):
        config = super().get_config(); config.update({'block_size': self.block_size}); return config


class ROAM(tf.keras.layers.Layer):
    """Ridge Orientation Attention Module."""
    def __init__(self, reduction=4, use_channel_gate=True, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
        # FIX: use_channel_gate must be a real constructor arg that build()/call()
        # branch on. Previously callers tried to monkeypatch `roam.channel_excite`
        # with a Lambda *before* the first call -- but build() runs on that first
        # call and unconditionally overwrote it with a trainable Dense layer, so
        # the "no_roam_channel_gate" ablation silently trained an unmodified model.
        self.use_channel_gate = use_channel_gate

    def build(self, input_shapes):
        feature_shape, _ = input_shapes
        channels = int(feature_shape[-1])
        reduced = max(channels // self.reduction, 8)
        self.orientation_proj = tf.keras.layers.Conv2D(reduced, 3, padding='same', activation='relu')
        self.spatial_gate = tf.keras.layers.Conv2D(1, 1, padding='same', activation='sigmoid')
        if self.use_channel_gate:
            self.channel_squeeze = tf.keras.layers.Dense(reduced, activation='relu')
            self.channel_excite = tf.keras.layers.Dense(channels, activation='sigmoid')
        self.resize_target = (int(feature_shape[1]), int(feature_shape[2]))

    def call(self, inputs):
        features, orientation_field = inputs
        orientation_resized = tf.image.resize(orientation_field, self.resize_target, method='bilinear')
        spatial_attention = self.spatial_gate(self.orientation_proj(orientation_resized))
        if self.use_channel_gate:
            channel_stats = tf.reduce_mean(features, axis=[1, 2])
            channel_attention = self.channel_excite(self.channel_squeeze(channel_stats))[:, None, None, :]
        else:
            channel_attention = 1.0  # true no-op: skips the gate entirely instead of shape-mismatched ones_like
        attended = features * spatial_attention * channel_attention
        return attended, spatial_attention

    def get_config(self):
        config = super().get_config()
        config.update({'reduction': self.reduction, 'use_channel_gate': self.use_channel_gate})
        return config


class AdaptiveGatedFusion(tf.keras.layers.Layer):
    def build(self, input_shapes):
        a_shape, b_shape = input_shapes
        dim = max(int(a_shape[-1]), int(b_shape[-1]))
        self.proj_a = tf.keras.layers.Dense(dim)
        self.proj_b = tf.keras.layers.Dense(dim)
        self.gate_dense = tf.keras.layers.Dense(dim, activation='sigmoid')

    def call(self, inputs):
        branch_a, branch_b = inputs
        a, b = self.proj_a(branch_a), self.proj_b(branch_b)
        gate = self.gate_dense(tf.concat([a, b], axis=-1))
        return gate * a + (1.0 - gate) * b, gate


In [9]:
# =========================
# RidgeVisionNet builder (+ ablation-variant builder)
# =========================

def build_ridgevision_net(img_size=IMG_SIZE, num_classes=NUM_CLASSES, dropout_rate=0.35,
                           trainable_backbone_layers=40, use_orientation_field=True,
                           use_channel_gate=True, fusion_mode='adaptive',
                           use_ridge_branch=True, use_appearance_branch=True, name='ridgevision_net'):
    assert use_ridge_branch or use_appearance_branch
    image_input = tf.keras.Input(shape=(img_size, img_size, 3), name='fingerprint_image')
    # FIX: tf.keras.applications.EfficientNetB0 has a built-in Rescaling(1/255)
    # layer expecting raw [0,255] pixels. Our shared pipeline already scales
    # images to [0,1] (Section 7.3), so without this correction EfficientNetB0
    # was dividing already-scaled pixels by 255 again -- crushing its input by
    # another factor of 255 and very likely causing the training collapses
    # observed in efficientnet_b0_plain, single_branch_texture, and
    # no_orientation_field. This undoes that scaling immediately before the
    # backbone; the grayscale/orientation-field path below is untouched since
    # it operates on the original [0,1] image_input, not this rescaled copy.
    efficientnet_input = tf.keras.layers.Rescaling(255.0, name='undo_pipeline_rescale_for_efficientnet')(image_input)
    backbone = load_backbone(tf.keras.applications.EfficientNetB0, weights='imagenet', include_top=False, input_tensor=efficientnet_input)
    for layer in backbone.layers:
        layer.trainable = False
    if trainable_backbone_layers > 0:
        for layer in backbone.layers[-trainable_backbone_layers:]:
            if not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True

    branches = []
    if use_ridge_branch:
        mid_features = backbone.get_layer('block6a_expand_activation').output
        if use_orientation_field:
            # Keras 3: raw tf.* ops cannot be applied directly to a KerasTensor
            # outside of a Layer; wrap in Lambda so it becomes a proper graph op.
            grayscale = tf.keras.layers.Lambda(
                lambda x: tf.image.rgb_to_grayscale(x), name='to_grayscale'
            )(image_input)
            orientation_field = RidgeOrientationField(block_size=8, dtype='float32')(grayscale)  # force float32: mixed precision would otherwise feed float16 into the hardcoded float32 Sobel kernels
        else:
            orientation_field = tf.keras.layers.Lambda(
                lambda x: tf.ones((tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], 3)),
                name='dummy_orientation_field'
            )(mid_features)
        roam = ROAM(use_channel_gate=use_channel_gate)
        attended_mid, spatial_attention = roam([mid_features, orientation_field])
        ridge_branch = tf.keras.layers.GlobalAveragePooling2D()(attended_mid)
        branches.append(ridge_branch)
    else:
        spatial_attention = None

    if use_appearance_branch:
        appearance_branch = tf.keras.layers.GlobalAveragePooling2D()(backbone.output)
        branches.append(appearance_branch)

    if len(branches) == 1:
        fused = branches[0]
    elif fusion_mode == 'adaptive':
        fused, _ = AdaptiveGatedFusion()(branches)
    elif fusion_mode == 'static_average':
        dim = 256
        projected = [tf.keras.layers.Dense(dim)(b) for b in branches]
        fused = tf.keras.layers.Average()(projected)
    elif fusion_mode == 'concat':
        fused = tf.keras.layers.Concatenate()(branches)
    else:
        raise ValueError(fusion_mode)

    hidden = tf.keras.layers.Dense(256, activation='relu')(fused)
    hidden = tf.keras.layers.Dropout(dropout_rate, name='mc_dropout')(hidden)
    logits = tf.keras.layers.Dense(num_classes, name='logits', dtype='float32')(hidden)
    probs = tf.keras.layers.Softmax(name='blood_group', dtype='float32')(logits)

    outputs = {'blood_group': probs, 'logits': logits}
    if spatial_attention is not None:
        outputs['attention_map'] = spatial_attention
    return tf.keras.Model(inputs=image_input, outputs=outputs, name=name)


ABLATION_GRID = [
    dict(name='full'),
    dict(name='no_orientation_field', use_orientation_field=False),
    dict(name='no_roam_channel_gate', use_channel_gate=False),
    dict(name='static_fusion', fusion_mode='static_average'),
    dict(name='concat_fusion', fusion_mode='concat'),
    dict(name='single_branch_texture', use_ridge_branch=False),
    dict(name='single_branch_ridge', use_appearance_branch=False),
    dict(name='no_finetune', trainable_backbone_layers=0),
]
print(f'{len(ABLATION_GRID)} ablation variants configured.')


8 ablation variants configured.


## Confidence estimation: MC-Dropout + temperature scaling

In [10]:
def mc_dropout_predict(model, dataset_or_array, n_samples=MC_DROPOUT_SAMPLES):
    # Keras 3's Model.__call__ only accepts tensors/arrays, not a
    # tf.keras.utils.Sequence (PyDataset). Materialize the Sequence into a
    # single array of images up front (same images the Sequence would
    # yield, just gathered once) so repeated MC-Dropout forward passes work.
    if isinstance(dataset_or_array, tf.keras.utils.Sequence):
        dataset_or_array = np.concatenate(
            [dataset_or_array[i][0] for i in range(len(dataset_or_array))], axis=0
        )
    samples = []
    for _ in range(n_samples):
        out = model(dataset_or_array, training=True)
        probs = out['blood_group'] if isinstance(out, dict) else out
        samples.append(np.asarray(probs))
    samples = np.stack(samples, axis=0)
    mean_probs = samples.mean(axis=0)
    eps = 1e-9
    predictive_entropy = -np.sum(mean_probs * np.log(mean_probs + eps), axis=-1)
    per_sample_entropy = -np.sum(samples * np.log(samples + eps), axis=-1)
    mutual_information = predictive_entropy - per_sample_entropy.mean(axis=0)
    return mean_probs, predictive_entropy, mutual_information


def fit_temperature(logits, labels):
    labels = labels.astype(int)
    def _nll(t):
        t = max(t, 1e-3)
        scaled = logits / t
        scaled -= scaled.max(axis=1, keepdims=True)
        log_probs = scaled - np.log(np.sum(np.exp(scaled), axis=1, keepdims=True))
        return -np.mean(log_probs[np.arange(len(labels)), labels])
    result = minimize_scalar(_nll, bounds=(0.05, 10.0), method='bounded')
    return float(result.x)


def apply_temperature(logits, temperature):
    scaled = logits / temperature
    scaled -= scaled.max(axis=1, keepdims=True)
    exp = np.exp(scaled)
    return exp / exp.sum(axis=1, keepdims=True)


def expected_calibration_error(confidences, correct, n_bins=15):
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, n = 0.0, len(confidences)
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(correct[mask].mean() - confidences[mask].mean())
    return float(ece)


def brier_score(probs, labels, n_classes=NUM_CLASSES):
    one_hot = np.eye(n_classes)[labels]
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))


## Training utility (shared recipe across RidgeVisionNet, baselines, and ablations)

In [11]:
def train_model(model, train_seq, val_seq, epochs_head=EPOCHS_HEAD, epochs_fine=EPOCHS_FINE,
                 class_weight=None, is_dict_output=True):
    output_name = 'blood_group' if is_dict_output else None
    loss = {output_name: 'sparse_categorical_crossentropy'} if is_dict_output else 'sparse_categorical_crossentropy'
    metrics = {output_name: 'accuracy'} if is_dict_output else ['accuracy']
    monitor = 'val_loss'  # verified key name for both dict-output and plain models

    # EarlyStopping cuts epochs once val loss stops improving (patience=6) and
    # restores the best-seen weights. This does not change the training
    # recipe's intent (same optimizer/LR schedule) -- it just stops paying for
    # epochs that were already flat/plateaued or had started overfitting,
    # which is where most of the wasted time in a fixed 15+40-epoch schedule
    # goes once a model has converged (or gotten stuck, as with mobilenet_v2).
    early_stop = tf.keras.callbacks.EarlyStopping(monitor=monitor, patience=7, restore_best_weights=True)
    # patience=7 (raised from 4): the real run showed resnet50-style architectures can plateau
    # 10+ epochs before a late breakthrough. patience=4 was cutting that breakthrough off before
    # it happened -- correctness matters more than shaving GPU time here.

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss, metrics=metrics)
    model.fit(train_seq, validation_data=val_seq, epochs=epochs_head, class_weight=class_weight,
              callbacks=[early_stop], verbose=2)

    # unfreeze already-configured trainable layers and fine-tune at low LR
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=loss, metrics=metrics)
    model.fit(train_seq, validation_data=val_seq, epochs=epochs_fine, class_weight=class_weight,
              callbacks=[early_stop], verbose=2)
    return model


def evaluate_model(model, test_seq, labels, is_dict_output=True):
    preds = model.predict(test_seq, verbose=0)
    probs = preds['blood_group'] if is_dict_output else preds
    y_pred = probs.argmax(axis=1)
    acc = accuracy_score(labels, y_pred)
    return acc, probs, y_pred


# =========================
# TUNED TRAINING HELPER (adds configurable head_lr/finetune_lr)
# =========================
# train_model() above hardcodes Adam(1e-3) / Adam(1e-5). MobileNetV2 and
# ResNet50 were independently confirmed (LR-search experiment) to collapse to
# near-chance accuracy at that default head LR but train normally at a lower
# one -- unrelated to the EfficientNetB0 rescaling bug fixed elsewhere in this
# notebook. This variant exposes the LR so the baseline-comparison loop can
# use a per-method override for those two specifically, while every other
# method keeps the original, unmodified recipe.
def train_model_tuned(model, train_seq, val_seq, epochs_head=EPOCHS_HEAD, epochs_fine=EPOCHS_FINE,
                       head_lr=1e-3, finetune_lr=1e-5, class_weight=None, is_dict_output=True, patience=7):
    output_name = 'blood_group' if is_dict_output else None
    loss = {output_name: 'sparse_categorical_crossentropy'} if is_dict_output else 'sparse_categorical_crossentropy'
    metrics = {output_name: 'accuracy'} if is_dict_output else ['accuracy']
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

    model.compile(optimizer=tf.keras.optimizers.Adam(head_lr), loss=loss, metrics=metrics)
    model.fit(train_seq, validation_data=val_seq, epochs=epochs_head, class_weight=class_weight,
              callbacks=[early_stop], verbose=2)

    model.compile(optimizer=tf.keras.optimizers.Adam(finetune_lr), loss=loss, metrics=metrics)
    model.fit(train_seq, validation_data=val_seq, epochs=epochs_fine, class_weight=class_weight,
              callbacks=[early_stop], verbose=2)
    return model

# Per-baseline LR overrides confirmed via independent LR-search experiment.
# Methods not listed here use the original default (1e-3, 1e-5) recipe.
BASELINE_LR_OVERRIDES = {
    'mobilenet_v2': (3e-4, 3e-6),
    'resnet50': (3e-5, 3e-7),
}


## 2. Ablation study (Table 7.2)

Trains each of the 8 configurations once on a single stratified split (not
5-fold, to keep total compute bounded -- rerun with folds if compute allows).

In [12]:
train_idx, test_idx = train_test_split(np.arange(len(all_paths_arr)), test_size=0.15, stratify=all_labels, random_state=SEED)
train_idx, val_idx = train_test_split(train_idx, test_size=0.1765, stratify=all_labels[train_idx], random_state=SEED)  # ~15% of total

train_paths, train_labels = all_paths_arr[train_idx], all_labels[train_idx]
val_paths, val_labels = all_paths_arr[val_idx], all_labels[val_idx]
test_paths, test_labels = all_paths_arr[test_idx], all_labels[test_idx]
class_weight_vals = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_labels)
class_weight = {i: w for i, w in enumerate(class_weight_vals)}

train_seq = FingerprintSequence(train_paths, train_labels, IMG_SIZE, augment=True)
val_seq = FingerprintSequence(val_paths, val_labels, IMG_SIZE, augment=False, shuffle=False)
test_seq = FingerprintSequence(test_paths, test_labels, IMG_SIZE, augment=False, shuffle=False)

FULL_MODEL_WEIGHTS_PATH = RESULTS_DIR / 'ridgevision_full_model.weights.h5'
ABLATION_CHECKPOINT_PATH = RESULTS_DIR / 'ablation_results.json'

# RESUME SUPPORT, same pattern as the baseline-comparison cell: reload any
# ablation results already written to disk and skip variants already done,
# so an interrupted/crashed session doesn't have to restart from scratch.
if ABLATION_CHECKPOINT_PATH.exists():
    with open(ABLATION_CHECKPOINT_PATH) as f:
        ablation_results = json.load(f)
    print(f'Resuming ablation from checkpoint: {ABLATION_CHECKPOINT_PATH}')
else:
    ablation_results = {}

def save_ablation_checkpoint():
    with open(ABLATION_CHECKPOINT_PATH, 'w') as f:
        json.dump(ablation_results, f, indent=2)

for cfg in ABLATION_GRID:
    name = cfg['name']
    if name in ablation_results:
        print(f'--- Ablation: {name} already done -- skipping. ---')
        continue

    kwargs = {k: v for k, v in cfg.items() if k != 'name'}
    print(f'\n--- Ablation: {name} ---')
    model = build_ridgevision_net(name=f'ablation_{name}', **kwargs)
    train_model(model, train_seq, val_seq, class_weight=class_weight)
    acc, probs, y_pred = evaluate_model(model, test_seq, test_labels)
    ablation_results[name] = {'test_accuracy': float(acc)}
    print(f'{name} test accuracy: {acc:.4f}')

    if name == 'full':
        # IMPORTANT (fixes the OOM/kernel-death crash from the last run):
        # do NOT keep a live Python reference to this model across the rest
        # of the loop. clear_session() below invalidates the session this
        # model's tensors belong to, but a live reference to it still stops
        # Python from freeing that GPU memory -- so it silently accumulates
        # underneath every subsequent ablation variant until the GPU runs
        # out. Save weights to disk instead; rebuild + reload them in the
        # Calibration/Robustness cells below, exactly like every other
        # variant gets deleted and rebuilt.
        model.save_weights(str(FULL_MODEL_WEIGHTS_PATH))
        print(f'Saved full model weights to {FULL_MODEL_WEIGHTS_PATH}')

    del model
    gc.collect(); tf.keras.backend.clear_session()
    save_ablation_checkpoint()

full_acc = ablation_results['full']['test_accuracy']
for name, entry in ablation_results.items():
    entry['delta_vs_full'] = entry['test_accuracy'] - full_acc
save_ablation_checkpoint()

for name, entry in ablation_results.items():
    print(f"{name:24s} acc={entry['test_accuracy']:.4f}  delta={entry.get('delta_vs_full', 0):+.4f}")



--- Ablation: full ---


I0000 00:00:1783864560.849778      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783864560.852847      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/8


2026-07-12 13:56:56.333970: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 13:56:56.494833: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 13:56:56.668460: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 13:56:56.833327: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1783864637.643182      73 device_compiler.h:196]

128/128 - 271s - 2s/step - loss: 1.3174 - val_loss: 0.5854
Epoch 2/8
128/128 - 27s - 208ms/step - loss: 0.6609 - val_loss: 0.4354
Epoch 3/8
128/128 - 25s - 198ms/step - loss: 0.5298 - val_loss: 0.4147
Epoch 4/8
128/128 - 25s - 193ms/step - loss: 0.4291 - val_loss: 0.3598
Epoch 5/8
128/128 - 25s - 198ms/step - loss: 0.3842 - val_loss: 0.3601
Epoch 6/8
128/128 - 26s - 201ms/step - loss: 0.3925 - val_loss: 0.3645
Epoch 7/8
128/128 - 26s - 202ms/step - loss: 0.3578 - val_loss: 0.3389
Epoch 8/8
128/128 - 24s - 191ms/step - loss: 0.3501 - val_loss: 0.3485
Epoch 1/30
128/128 - 154s - 1s/step - loss: 0.2624 - val_loss: 0.2890
Epoch 2/30
128/128 - 25s - 193ms/step - loss: 0.2261 - val_loss: 0.2790
Epoch 3/30
128/128 - 25s - 192ms/step - loss: 0.2148 - val_loss: 0.2837
Epoch 4/30
128/128 - 25s - 194ms/step - loss: 0.2065 - val_loss: 0.2764
Epoch 5/30
128/128 - 25s - 196ms/step - loss: 0.2080 - val_loss: 0.2747
Epoch 6/30
128/128 - 27s - 208ms/step - loss: 0.1909 - val_loss: 0.2701
Epoch 7/30
128

2026-07-12 14:15:30.507689: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 14:15:30.651119: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 14:15:31.096100: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 14:15:31.238468: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 14:15:32.101042: E external/local_xla/xla/stream_

128/128 - 108s - 841ms/step - loss: 1.2828 - val_loss: 0.4994
Epoch 2/8
128/128 - 25s - 193ms/step - loss: 0.6392 - val_loss: 0.4458
Epoch 3/8
128/128 - 24s - 191ms/step - loss: 0.5394 - val_loss: 0.4445
Epoch 4/8
128/128 - 24s - 189ms/step - loss: 0.4645 - val_loss: 0.5226
Epoch 5/8
128/128 - 25s - 194ms/step - loss: 0.3759 - val_loss: 0.4027
Epoch 6/8
128/128 - 25s - 193ms/step - loss: 0.3561 - val_loss: 0.3713
Epoch 7/8
128/128 - 25s - 191ms/step - loss: 0.3641 - val_loss: 0.3560
Epoch 8/8
128/128 - 25s - 194ms/step - loss: 0.3241 - val_loss: 0.3134
Epoch 1/30
128/128 - 87s - 683ms/step - loss: 0.2164 - val_loss: 0.2856
Epoch 2/30
128/128 - 26s - 200ms/step - loss: 0.1993 - val_loss: 0.2735
Epoch 3/30
128/128 - 25s - 197ms/step - loss: 0.1865 - val_loss: 0.2676
Epoch 4/30
128/128 - 25s - 196ms/step - loss: 0.1725 - val_loss: 0.2667
Epoch 5/30
128/128 - 25s - 193ms/step - loss: 0.1669 - val_loss: 0.2655
Epoch 6/30
128/128 - 25s - 195ms/step - loss: 0.1723 - val_loss: 0.2636
Epoch 7/3

## 3. Calibration: MC-Dropout + temperature scaling (feeds Section 4.4 / 7.3)

Uses the `full` ablation model retrained just above.

In [13]:
# Rebuild the 'full' RidgeVisionNet architecture fresh and load the weights
# saved during the ablation loop above, rather than relying on a live Python
# reference held across many other model builds (that pattern is exactly
# what caused the OOM/kernel-death crash previously -- see the ablation cell).
full_model = build_ridgevision_net(name='full_reloaded_for_calibration')
full_model.load_weights(str(FULL_MODEL_WEIGHTS_PATH))
print('Reloaded full model weights from', FULL_MODEL_WEIGHTS_PATH)


Reloaded full model weights from /kaggle/working/ridgevisionnet_results/ridgevision_full_model.weights.h5


In [14]:
# Re-split validation set purely for temperature fitting (held out from test)
val_logits = full_model.predict(val_seq, verbose=0)['logits']
temperature = fit_temperature(val_logits, val_labels)
print('Fitted temperature T =', temperature)

test_logits = full_model.predict(test_seq, verbose=0)['logits']
raw_probs = tf.nn.softmax(test_logits, axis=-1).numpy()
calibrated_probs = apply_temperature(test_logits, temperature)

y_pred = calibrated_probs.argmax(axis=1)
correct = (y_pred == test_labels).astype(float)

raw_conf = raw_probs.max(axis=1)
cal_conf = calibrated_probs.max(axis=1)

ece_raw = expected_calibration_error(raw_conf, correct)
ece_cal = expected_calibration_error(cal_conf, correct)
brier_raw = brier_score(raw_probs, test_labels)
brier_cal = brier_score(calibrated_probs, test_labels)

mean_probs, pred_entropy, mutual_info = mc_dropout_predict(full_model, test_seq)

calibration_results = {
    'temperature': temperature,
    'ece_before_calibration': ece_raw,
    'ece_after_calibration': ece_cal,
    'brier_before_calibration': brier_raw,
    'brier_after_calibration': brier_cal,
    'mc_dropout_mean_predictive_entropy': float(pred_entropy.mean()),
    'mc_dropout_mean_mutual_information': float(mutual_info.mean()),
}
with open(RESULTS_DIR / 'calibration_results.json', 'w') as f:
    json.dump(calibration_results, f, indent=2)
print(json.dumps(calibration_results, indent=2))


Fitted temperature T = 1.3645357803610998


ResourceExhaustedError: Exception encountered when calling Conv2D.call().

[1m{{function_node __wrapped__Conv2D_device_/job:localhost/replica:0/task:0/device:GPU:0}} OOM when allocating tensor with shape[876,112,112,16] and type float on /job:localhost/replica:0/task:0/device:GPU:0 by allocator GPU_0_bfc [Op:Conv2D][0m

Arguments received by Conv2D.call():
  • inputs=tf.Tensor(shape=(876, 112, 112, 32), dtype=float32)

---
### Part 2 (v2) done.

Save Version -> Save & Run All (Commit), then create/update a Kaggle Dataset
from the output. Attach it as Input to Part 3 (v2).
